In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime
from pyspark.sql.window import Window
import uuid


In [0]:
%sql
use catalog novacart_workspace;
create schema if not exists gold_schema;

In [0]:
gold_run_id=uuid.uuid4()


run_ts_str=datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')
run_date_str=datetime.utcnow().strftime('%Y-%m-%d')

print("current Gold Run ID:", gold_run_id)
print("Run Timestamp Folder:", run_ts_str)


                                        


In [0]:
%sql
create table if not exists novacart_workspace.gold_schema.processing_control(
layer string,
entity_name string,
last_processed_silver_run_id string,
last_processed_silver_run_ts timestamp,
rows_merged bigint,
run_status string,
gold_run_id string,
updated_at timestamp

)

In [0]:

from delta.tables import DeltaTable

def upsert_to_gold(df_source, target_table, join_key):

    if spark.catalog.tableExists(target_table):

        target = DeltaTable.forName(spark, target_table)

        (
            target.alias("target")
            .merge(
                df_source.alias("source"),
                f"target.{join_key} = source.{join_key}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    else:
        (
            df_source.write
            .format("delta")
            .saveAsTable(target_table)
        )


In [0]:
%sql
select * from novacart_workspace.gold_schema.processing_control

In [0]:
def  get_last_processed_silver_ts(entity_name):
    ctrl=(
        spark.table("novacart_workspace.gold_schema.processing_control").filter(
            (col("layer") == "gold") &
            (col("entity_name")==entity_name)&
            (col("run_status") == 'SUCCESS')
        ).orderBy(col("updated_at").desc()).limit(1)
    )
    rows=ctrl.collect()
    if not rows:
        return None
    else:
        return rows[0] ["last_processed_silver_run_ts"]

In [0]:

from datetime import datetime
from delta.tables import DeltaTable

def upsert_gold_control(
    entity_name,
    last_processed_silver_run_id,
    last_processed_silver_run_ts,
    rows_merged
):
    ctrl_df = spark.createDataFrame(
        [(
            "gold",
            entity_name,
            last_processed_silver_run_id,
            last_processed_silver_run_ts,
            int(rows_merged),
            "SUCCESS",
            gold_run_id,
            datetime.utcnow()
        )],
        schema="""
            layer STRING,
            entity_name STRING,
            last_processed_silver_run_id STRING,
            last_processed_silver_run_ts TIMESTAMP,
            rows_merged BIGINT,
            run_status STRING,
            gold_run_id STRING,
            updated_at TIMESTAMP
        """
    )

    df = DeltaTable.forName(
        spark,
        "novacart_workspace.gold_schema.processing_control"
    )

    (
        df.alias("t")
        .merge(
            ctrl_df.alias("s"),
            "t.layer = s.layer AND t.entity_name = s.entity_name"
        )
        .whenMatchedUpdate(set={
            "last_processed_silver_run_id":
                "s.last_processed_silver_run_id",
            "last_processed_silver_run_ts":
                "s.last_processed_silver_run_ts",
            "rows_merged": "s.rows_merged",
            "run_status": "s.run_status",
            "gold_run_id": "s.gold_run_id",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )


In [0]:
last_gold_ts=get_last_processed_silver_ts("orders_information")
print("Last Processed Silver Timestamp for Gold=",last_gold_ts)

silver_orders_current = spark.read.table("novacart_workspace.silver_schema.orders_transformed")

silver_products_current = spark.read.table("novacart_workspace.silver_schema.products_transformed")

silver_payments_current = spark.read.table("novacart_workspace.silver_schema.payments_transformed")

if last_gold_ts is None:
    changed_orders = silver_orders_current
    changed_products = silver_products_current 
    changed_payments = silver_payments_current
else:
    changed_orders = silver_orders_current.filter(col("updated_at")>last_gold_ts)
    changed_products = silver_products_current.filter(col("updated_at")>last_gold_ts)
    changed_payments = silver_payments_current.filter(col("updated_at")>last_gold_ts)

changed_orders_count = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()

print("Number of changed orders=",changed_orders_count)
print("Number of changed products=",changed_products_count)
print("Number of changed payments=",changed_payments_count)







In [0]:
impacted_from_orders = changed_orders.select("order_id").distinct()
print(impacted_from_orders)

In [0]:
impacted_from_orders = changed_orders.select("order_id").distinct()
impacted_from_payments = changed_payments.select("order_id").distinct()
impacted_from_products = (
    changed_products.alias("P").join(silver_orders_current.alias("O"), col("P.product_id")==col("O.product_id"), "inner")
    .select("O.order_id").distinct()
)

impacted_order_ids=(
    impacted_from_orders.union(impacted_from_payments).union(impacted_from_products).select("order_id")
    .distinct()
)

print("impacted_orders_ids = ", impacted_order_ids.count())
display(impacted_order_ids.orderBy("order_id"))

In [0]:
impacted_orders = (
    silver_orders_current.alias("o")
    .join(impacted_order_ids.alias("i"), "order_id", "inner")
)

gold_delta = (
    impacted_orders.alias("o")
    .join(
        silver_products_current.alias("p"),
        col("o.product_id") == col("p.product_id"),
        "inner"
    )
    .join(
        silver_payments_current.alias("py"),
        col("o.order_id") == col("py.order_id"),
        "inner"
    )
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("p.product_id"),
        col("p.product_name"),
        col("p.category"),
        col("p.price"),
        col("o.order_status"),
        col("o.order_amount"),
        col("py.payment_id"),
        col("py.payment_status"),
        col("py.paid_amount"),
        col("o.order_date"),
        col("o.order_month"),
        col("o.order_year"),
        greatest(
            col("o.updated_at"),
            col("p.updated_at"),
            col("py.processed_at")
        ).alias("gold_update_ts")
    )
    .dropDuplicates(["order_id"])
    .withColumn(
        "payment_completed_ratio",
        when(
            col("order_amount") > 0,
            col("paid_amount") / col("order_amount")
        ).otherwise(lit(0.0))
    )
    .withColumn(
        "payment_state",
        when(col("order_amount") == 0, "Invalid_order_amount")
        .when(col("payment_completed_ratio") <= 0, "Unpaid")
        .when(col("payment_completed_ratio") == 1, "Paid")
        .when(col("payment_completed_ratio") < 1, "Partially_paid")
        .when(col("payment_completed_ratio") > 1, "Overpaid")
    )
    .withColumn(
        "gold_updated_date",
        to_date(col("gold_update_ts"))
    )
    .withColumn(
        "gold_run_id",
        lit(str(gold_run_id))
    )
)

print("gold_delta_rows", gold_delta.count())

display(gold_delta)

In [0]:
if gold_delta.count()>0:
    upsert_to_gold(gold_delta,"novacart_workspace.gold_schema.orders_information","order_id")
else:
    print("No new rows to insert in gold table")

In [0]:
%sql
select * from novacart_workspace.gold_schema.orders_information

In [0]:
if not spark.catalog.tableExists("novacart_workspace.gold_schema.orders_information_scd2"):
    spark.sql(""" 
              create table if not exists novacart_workspace.gold_schema.orders_information_scd2
              using delta as
              select *,cast(null as timestamp) as vaild_from_ts,
              cast(null as timestamp) as vaild_to_ts,
              true as is_current
              from novacart_workspace.gold_schema.orders_information
              where 1=0
              """)
if gold_delta.count()>0:
    gold_delta.createOrReplaceTempView("gold_delta_view")
    spark.sql(""" 
              merge into novacart_workspace.gold_schema.orders_information_scd2 t using gold_delta_view s
              on t.order_id=s.order_id and t.is_current = true
              when matched and(
                  not (t.order_status <=> s.order_status) or
                  not(t.order_amount <=> s.order_amount) or
                  not(t.paid_amount <=> s.paid_amount) or
                  not(t.payment_id <=> s.payment_id) or
                  not(t.category <=> s.category) or
                  not(t.product_name <=> s.product_name) or
                  not(t.price <=> s.price))
                  then update set t.is_current=false,
                  t.vaild_to_ts=s.gold_update_ts
              
              """)
    spark.sql("""
              insert into novacart_workspace.gold_schema.orders_information_scd2
              select s.*,s.gold_update_ts as vaild_from_ts,
              cast(null as timestamp) as vaild_to_ts,
              true as is_current 
              from gold_delta_view s
              left join novacart_workspace.gold_schema.orders_information_scd2 t
              on s.order_id=t.order_id and t.is_current=true
              where t.order_id is null or (
                  not(t.order_status <=> s.order_status) or 
                  not(t.order_amount <=> s.order_amount) or
                  not(t.paid_amount <=> s.paid_amount) or
                  not(t.payment_id <=> s.payment_id) or
                  not(t.category <=> s.category) or
                  not(t.product_name <=> s.product_name) or
                  not(t.price <=> s.price)
              )

              """) 

In [0]:
if gold_delta.count()>0:
    impacted_categories =(
        gold_delta.select("category").filter(col("category").isNotNull()).distinct()
    )
    category_perf_delta = (
    spark.read.table("novacart_workspace.gold_schema.orders_information" )
    .join(impacted_categories, "category", "inner")
    .groupBy("category")
    .agg(
        countDistinct("order_id").alias("total_orders"),

        sum(
            when(col("order_amount") > 0, col("order_amount"))
            .otherwise(lit(0.0))
        ).alias("Gross_Merchandise_Value"),

        sum(
            when(col("paid_amount") > 0, col("paid_amount"))
            .otherwise(lit(0.0))
        ).alias("Total_Paid_Amount"),

        avg(
            col("payment_completed_ratio")
        ).alias("Average_Payment_Completion_Ratio"),

        (
            sum(
                when(col("payment_status") == "FAILED", 1)
                .otherwise(0)
            ) / count("*")
        ).alias("Payment_Failure_Rate")
    )
)

upsert_to_gold(
    category_perf_delta,
    "novacart_workspace.gold_schema.category_performance",
    "category"
)

In [0]:
spark.sql("create volume if not exists novacart_workspace.gold_schema.gold_snapshot")
latest_orders_path=('/Volumes/novacart_workspace/gold_schema/gold_snapshot/gold_latest/order_information')
latest_category_path=('/Volumes/novacart_workspace/gold_schema/gold_snapshot/gold_latest/category_performance')
historical_orders_path=(f'/Volumes/novacart_workspace/gold_schema/gold_snapshot/gold_snapshots/orders_information/run_date={run_date_str}/run_ts={run_ts_str}')
historical_category_path=(f'/Volumes/novacart_workspace/gold_schema/gold_snapshot/gold_snapshots/category_performance/run_date={run_date_str}/run_ts={run_ts_str}')

spark.read.table("novacart_workspace.gold_schema.orders_information").write.mode("overwrite").format("parquet").save(latest_orders_path)

spark.read.table("novacart_workspace.gold_schema.category_performance").write.mode("overwrite").format("parquet").save(latest_category_path)

spark.read.table("novacart_workspace.gold_schema.orders_information").write.mode("overwrite").format("parquet").save(historical_orders_path)

spark.read.table("novacart_workspace.gold_schema.category_performance").write.mode("overwrite").format("parquet").save(historical_category_path)

print("latest orders path :",latest_orders_path)
print("latest category path :",latest_category_path)
print("historical orders path :",historical_orders_path)
print("historical category path :",historical_category_path)




In [0]:
from pyspark.sql.functions import col, max

latest_silver_ts = (
    silver_orders_current
    .agg(max("bronze_ingested_at").alias("mx"))
    .collect()[0]["mx"]
)

latest_silver_run_id = (
    silver_orders_current
    .filter(col("bronze_ingested_at") == latest_silver_ts)
    .agg(max("silver_run_id").alias("mx"))
    .collect()[0]["mx"]
    if latest_silver_ts is not None
    else None
)

upsert_gold_control(
    "orders_information",
    latest_silver_run_id,
    latest_silver_ts,
    gold_delta.count()
)

display(
    spark.table("novacart_workspace.gold_schema.processing_control")
)